In [1]:
import os
import pandas as pd
import pyarrow

In [2]:
df = pd.read_csv('data/raw/flight_data_2024.csv')
df.info()

/var/folders/0p/sx2q3w3d0n12d_0t4x5g96m80000gn/T/ipykernel_66969/3346776205.py:1: DtypeWarning: Columns (24) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/raw/flight_data_2024.csv')


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7079081 entries, 0 to 7079080
Data columns (total 35 columns):
 #   Column               Dtype  
---  ------               -----  
 0   year                 int64  
 1   month                int64  
 2   day_of_month         int64  
 3   day_of_week          int64  
 4   fl_date              object 
 5   op_unique_carrier    object 
 6   op_carrier_fl_num    float64
 7   origin               object 
 8   origin_city_name     object 
 9   origin_state_nm      object 
 10  dest                 object 
 11  dest_city_name       object 
 12  dest_state_nm        object 
 13  crs_dep_time         int64  
 14  dep_time             float64
 15  dep_delay            float64
 16  taxi_out             float64
 17  wheels_off           float64
 18  wheels_on            float64
 19  taxi_in              float64
 20  crs_arr_time         int64  
 21  arr_time             float64
 22  arr_delay            float64
 23  cancelled            int64  
 24

In [3]:
df.head(15)

,year,month,day_of_month,day_of_week,fl_date,op_unique_carrier,op_carrier_fl_num,origin,origin_city_name,origin_state_nm,...,diverted,crs_elapsed_time,actual_elapsed_time,air_time,distance,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay
0,2024,1,1,1,2024-01-01,9E,4814.0,JFK,"New York, NY",New York,...,0,136.0,122.0,84.0,509.0,0,0,0,0,0
1,2024,1,1,1,2024-01-01,9E,4815.0,MSP,"Minneapolis, MN",Minnesota,...,0,130.0,114.0,88.0,622.0,0,0,0,0,0
2,2024,1,1,1,2024-01-01,9E,4817.0,JFK,"New York, NY",New York,...,0,106.0,90.0,61.0,288.0,0,0,0,0,0
3,2024,1,1,1,2024-01-01,9E,4817.0,RIC,"Richmond, VA",Virginia,...,0,111.0,76.0,51.0,288.0,0,0,0,0,0
4,2024,1,1,1,2024-01-01,9E,4818.0,DTW,"Detroit, MI",Michigan,...,0,79.0,70.0,45.0,237.0,0,0,0,0,0
5,2024,1,1,1,2024-01-01,9E,4822.0,JAX,"Jacksonville, FL",Florida,...,0,137.0,120.0,102.0,833.0,0,0,0,0,0
6,2024,1,1,1,2024-01-01,9E,4822.0,LGA,"New York, NY",New York,...,0,169.0,164.0,125.0,833.0,0,0,0,0,0
7,2024,1,1,1,2024-01-01,9E,4823.0,CHS,"Charleston, SC",South Carolina,...,0,118.0,99.0,86.0,641.0,0,0,0,0,0
8,2024,1,1,1,2024-01-01,9E,4823.0,LGA,"New York, NY",New York,...,0,149.0,123.0,101.0,641.0,0,0,0,0,0
9,2024,1,1,1,2024-01-01,9E,4828.0,ITH,"Ithaca/Cortland, NY",New York,...,0,79.0,67.0,43.0,189.0,0,0,0,0,0


In [4]:
print(f"Record iniziali: {len(df)}")

Record iniziali: 7079081


**Pulizia dei voli cancellati e deviati**

In [5]:
print(df['cancelled'].unique())
print(df['cancelled'].isnull().sum())

[0 1]
0


In [6]:
print(df['diverted'].unique())
print(df['diverted'].isnull().sum())

[0 1]
0


Tutto bene, non ci sono valori nulli o valori sporchi

Adesso si crea una maschera booleana, per identificare gli eventuali record corrotti: voli regolari ma con ritardi mancanti

In [7]:
maschere = (df['cancelled'] == 0) & (df['diverted'] == 0) & (df['arr_delay'].isna() | df['dep_delay'].isna())
df_clean = df[~maschere].copy()
print(f"Record dopo la prima pulizia: {len(df_clean)}")

Record dopo la prima pulizia: 7079081


In [8]:
print(df['year'].unique())

[2024]


Cancello la colonna year: contiene sempre lo stesso campo, perciò è del tutto superfluo

**Parsing della data**

In [9]:
df_clean = df_clean.drop(columns=['year'])

In [10]:
df_clean['fl_date'] = pd.to_datetime(df_clean['fl_date'])

**Selezione delle colonne target**

In [11]:
target = ['month', 'fl_date', 
    'op_unique_carrier', 'op_carrier_fl_num', 
    'origin', 'dest', 
    'dep_delay', 'arr_delay', 
    'cancelled', 'cancellation_code',
    'carrier_delay', 'weather_delay', 'nas_delay', 'security_delay', 'late_aircraft_delay']

# Per evitare errori
esistenti = [colonna for colonna in target if colonna in df_clean.columns]
df_clean = df_clean[esistenti]

**Normalizzazione e imputazione per le cause di ritardo**

Un NaN in queste colonne significa "0 minuti di ritardo imputabili a questa causa"

In [12]:
cause = ['carrier_delay', 'weather_delay', 'nas_delay', 'security_delay', 'late_aircraft_delay']
for causa in cause:
    if causa in df_clean.columns:
        df_clean[causa] = pd.to_numeric(df_clean[causa], errors='coerce').fillna(0)

In [13]:
print(f"Numero di colonne mantenute: {len(df_clean.columns)}")

Numero di colonne mantenute: 15


In [14]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7079081 entries, 0 to 7079080
Data columns (total 15 columns):
 #   Column               Dtype         
---  ------               -----         
 0   month                int64         
 1   fl_date              datetime64[ns]
 2   op_unique_carrier    object        
 3   op_carrier_fl_num    float64       
 4   origin               object        
 5   dest                 object        
 6   dep_delay            float64       
 7   arr_delay            float64       
 8   cancelled            int64         
 9   cancellation_code    object        
 10  carrier_delay        int64         
 11  weather_delay        int64         
 12  nas_delay            int64         
 13  security_delay       int64         
 14  late_aircraft_delay  int64         
dtypes: datetime64[ns](1), float64(3), int64(7), object(4)
memory usage: 810.1+ MB


In [15]:
df_clean.head(15)

,month,fl_date,op_unique_carrier,op_carrier_fl_num,origin,dest,dep_delay,arr_delay,cancelled,cancellation_code,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay
0,1,2024-01-01,9E,4814.0,JFK,DTW,-5.0,-19.0,0,NaN,0,0,0,0,0
1,1,2024-01-01,9E,4815.0,MSP,CLE,-14.0,-30.0,0,NaN,0,0,0,0,0
2,1,2024-01-01,9E,4817.0,JFK,RIC,-4.0,-20.0,0,NaN,0,0,0,0,0
3,1,2024-01-01,9E,4817.0,RIC,JFK,-7.0,-42.0,0,NaN,0,0,0,0,0
4,1,2024-01-01,9E,4818.0,DTW,MKE,-5.0,-14.0,0,NaN,0,0,0,0,0
5,1,2024-01-01,9E,4822.0,JAX,LGA,-7.0,-24.0,0,NaN,0,0,0,0,0
6,1,2024-01-01,9E,4822.0,LGA,JAX,-8.0,-13.0,0,NaN,0,0,0,0,0
7,1,2024-01-01,9E,4823.0,CHS,LGA,-5.0,-24.0,0,NaN,0,0,0,0,0
8,1,2024-01-01,9E,4823.0,LGA,CHS,-5.0,-31.0,0,NaN,0,0,0,0,0
9,1,2024-01-01,9E,4828.0,ITH,JFK,-12.0,-24.0,0,NaN,0,0,0,0,0


In [16]:
df_clean.isna().sum()

month                        0
fl_date                      0
op_unique_carrier            0
op_carrier_fl_num            1
origin                       0
dest                         0
dep_delay                92970
arr_delay               113814
cancelled                    0
cancellation_code      6982766
carrier_delay                0
weather_delay                0
nas_delay                    0
security_delay               0
late_aircraft_delay          0
dtype: int64

In [17]:
df_clean['op_carrier_fl_num'] = df_clean['op_carrier_fl_num'].dropna()

In [18]:
df_clean.isna().sum()

month                        0
fl_date                      0
op_unique_carrier            0
op_carrier_fl_num            1
origin                       0
dest                         0
dep_delay                92970
arr_delay               113814
cancelled                    0
cancellation_code      6982766
carrier_delay                0
weather_delay                0
nas_delay                    0
security_delay               0
late_aircraft_delay          0
dtype: int64

In [19]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7079081 entries, 0 to 7079080
Data columns (total 15 columns):
 #   Column               Dtype         
---  ------               -----         
 0   month                int64         
 1   fl_date              datetime64[ns]
 2   op_unique_carrier    object        
 3   op_carrier_fl_num    float64       
 4   origin               object        
 5   dest                 object        
 6   dep_delay            float64       
 7   arr_delay            float64       
 8   cancelled            int64         
 9   cancellation_code    object        
 10  carrier_delay        int64         
 11  weather_delay        int64         
 12  nas_delay            int64         
 13  security_delay       int64         
 14  late_aircraft_delay  int64         
dtypes: datetime64[ns](1), float64(3), int64(7), object(4)
memory usage: 810.1+ MB


In [20]:
limiti = {
    'dep_delay_min': df_clean['dep_delay'].min(),
    'dep_delay_max': df_clean['dep_delay'].max(),
    'arr_delay_min': df_clean['arr_delay'].min(),
    'arr_delay_max': df_clean['arr_delay'].max(),
    'ok_min': -30.0,
    'ok_max': 60.0*24
}
print(f"Valori anomali")
print(f"dep_delay: {limiti['dep_delay_min']} - {limiti['dep_delay_max']}")
print(f"arr_delay: {limiti['arr_delay_min']} - {limiti['arr_delay_max']}")

Valori anomali
dep_delay: -96.0 - 3777.0
arr_delay: -126.0 - 3803.0


In [21]:
outliers = df_clean.query("dep_delay < @limiti['ok_min'] or dep_delay > @limiti['ok_max'] or \
                           arr_delay < @limiti['ok_min'] or arr_delay > @limiti['ok_max']")
conta = len(outliers)
print(f"Conta celle valori anomali: {conta}")
print(f"Percentuale: {conta/len(df_clean)*100}%")

Conta celle valori anomali: 206367
Percentuale: 2.915166530796865%


In [22]:
df_clean = df_clean.drop(outliers.index)
print(f"Numero di record: {len(df_clean)}")

Numero di record: 6872714


In [23]:
df_clean.query("cancellation_code.isna() and cancelled == 1").isna().sum()

month                  0
fl_date                0
op_unique_carrier      0
op_carrier_fl_num      0
origin                 0
dest                   0
dep_delay              0
arr_delay              0
cancelled              0
cancellation_code      0
carrier_delay          0
weather_delay          0
nas_delay              0
security_delay         0
late_aircraft_delay    0
dtype: int64

In [24]:
percentuali = [0.10, 0.25, 0.50, 0.75, 1.0, 1.5]

for p in percentuali:
    label = int(p * 100)
    local_path = f"data/cleaned/dataset_{label}.parquet"
    local_path_csv = f"data/cleaned/dataset_{label}.csv"
    
    print(f"Generazione {label}%...")
    
    if p <= 1.0:
        df_sample = df_clean.sample(frac=p, replace=False, random_state=42)
        
    else:
        df_extra = df_clean.sample(frac=0.5, replace=False, random_state=42)
        df_sample = pd.concat([df_clean, df_extra])
    
    df_sample.to_parquet(local_path, engine='pyarrow', index=False)
    df_sample.to_csv(local_path_csv, index=False)
    print(f"Salvato: {local_path}")
    print(f"Salvato: {local_path_csv}")


Generazione 10%...
Salvato: data/cleaned/dataset_10.parquet
Salvato: data/cleaned/dataset_10.csv
Generazione 25%...
Salvato: data/cleaned/dataset_25.parquet
Salvato: data/cleaned/dataset_25.csv
Generazione 50%...
Salvato: data/cleaned/dataset_50.parquet
Salvato: data/cleaned/dataset_50.csv
Generazione 75%...
Salvato: data/cleaned/dataset_75.parquet
Salvato: data/cleaned/dataset_75.csv
Generazione 100%...
Salvato: data/cleaned/dataset_100.parquet
Salvato: data/cleaned/dataset_100.csv
Generazione 150%...
Salvato: data/cleaned/dataset_150.parquet
Salvato: data/cleaned/dataset_150.csv
